In [ ]:
import os, shutil, urllib.request

url = 'https://github.com/kmu-agent/data/raw/refs/heads/main/campaign_data.zip'
file_name = url.split("/")[-1]

if not os.path.exists(file_name):
    print("데이터 다운로드 중...")
    urllib.request.urlretrieve(url, file_name)
    shutil.unpack_archive(file_name, "./", "zip")
    print("완료")
else:
    print("데이터 파일이 이미 존재합니다.")

In [ ]:
import subprocess, sys
for pkg in ["scikit-learn", "xgboost", "catboost", "optuna", "lightgbm"]:
    try: __import__(pkg if pkg != "scikit-learn" else "sklearn")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
SEED = 42
DATA_PATH = "./"
N_SPLITS = 5
N_REPEATS = 3
N_TRIALS = 30
SUBMIT_PREFIX = "submit_머신러닝_표승렬"

In [69]:
train = pd.read_csv(f"{DATA_PATH}campaign_train.csv")
test  = pd.read_csv(f"{DATA_PATH}campaign_test.csv")
submit_template = pd.read_csv(f"{DATA_PATH}campaign_sample_submission.csv")

print("train:", train.shape, " test:", test.shape, " submit:", submit_template.shape)

target = train["target"]
train_x = train.drop(columns=["target"])
test_x  = test.copy()

train: (1344, 27)  test: (896, 26)  submit: (896, 2)


- 데이터 불러오기

In [70]:
def make_features(df, ref_date):
    df = df.copy()

    # 1. 가입날짜 분해
    df["고객_가입날짜"] = pd.to_datetime(df["고객_가입날짜"], errors="coerce")
    df["가입연도"] = df["고객_가입날짜"].dt.year
    df["가입월"]   = df["고객_가입날짜"].dt.month
    df["가입요일"] = df["고객_가입날짜"].dt.dayofweek
    df["가입기간"] = (ref_date - df["고객_가입날짜"]).dt.days
    df = df.drop(columns=["고객_가입날짜"])

    # 2. 나이 (ref_date 기준으로 통일 - 시점 일관성)
    df["고객_나이"] = ref_date.year - df["출생연도"]

    # 3. 가족 관련
    df["총_자녀수"] = df["고객_자녀수"] + df["고객_청소년수"]
    df["자녀_있음"] = (df["총_자녀수"] > 0).astype(int)

    # 4. 구매금액 관련
    amount_cols = [
        "고객_와인_구매금액","고객_과일_구매금액","고객_육류_구매금액",
        "고객_생선_구매금액","고객_사탕_구매금액","고객_골드_구매금액"
    ]
    df["총_구매금액"]       = df[amount_cols].sum(axis=1)
    df["평균_품목구매금액"] = df[amount_cols].mean(axis=1)
    df["구매금액_표준편차"] = df[amount_cols].std(axis=1)
    for col in amount_cols:
        df[col + "_비중"] = df[col] / (df["총_구매금액"] + 1)

    # 5. 구매횟수 관련
    count_cols = [
        "고객_할인품목_구매횟수","고객_회사사이트_통한_구매횟수",
        "고객_카탈로그_통한_구매횟수","고객_매장방문_구매횟수"
    ]
    df["총_구매횟수"]   = df[count_cols].sum(axis=1)
    df["평균_구매금액"] = df["총_구매금액"] / (df["총_구매횟수"] + 1)
    df["할인구매_비중"]     = df["고객_할인품목_구매횟수"]     / (df["총_구매횟수"] + 1)
    df["웹구매_비중"]       = df["고객_회사사이트_통한_구매횟수"] / (df["총_구매횟수"] + 1)
    df["카탈로그구매_비중"] = df["고객_카탈로그_통한_구매횟수"] / (df["총_구매횟수"] + 1)
    df["매장구매_비중"]     = df["고객_매장방문_구매횟수"]     / (df["총_구매횟수"] + 1)

    # 6. 소득 대비 구매력
    df["소득대비_구매금액"]   = df["총_구매금액"]   / (df["고객_소득"] + 1)
    df["소득대비_평균구매금액"] = df["평균_구매금액"] / (df["고객_소득"] + 1)

    # 7. 웹 방문 효율
    df["웹방문대비_웹구매"] = (
        df["고객_회사사이트_통한_구매횟수"]
        / (df["고객_지난달_회사사이트_방문횟수"] + 1)
    )

    # 8. 과거 캠페인 반응
    campaign_cols = [f"캠페인{i}_수락여부" for i in range(1, 6)]
    df["과거캠페인_수락횟수"] = df[campaign_cols].sum(axis=1)
    df["과거캠페인_수락경험"] = (df["과거캠페인_수락횟수"] > 0).astype(int)

    # 9. ID 제거
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    return df

In [ ]:
ref_date = pd.Timestamp("2024-01-01")  # 재현성을 위한 고정 기준 날짜

train_ft = make_features(train_x, ref_date)
test_ft  = make_features(test_x,  ref_date)

print("train_ft:", train_ft.shape, " test_ft:", test_ft.shape)

In [71]:
cat_cols = train_ft.select_dtypes(include="object").columns.tolist()
num_cols = train_ft.select_dtypes(exclude="object").columns.tolist()
print("범주형:", cat_cols)
print("수치형 개수:", len(num_cols))

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop"
)

범주형: ['고객_교육수준', '고객_결혼여부']
수치형 개수: 49


In [72]:
def run_cv(model_factory, fit_kwargs_factory=None, verbose=True):

    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    n_total = N_SPLITS * N_REPEATS

    oof  = np.zeros(len(train_ft))
    pred = np.zeros(len(test_ft))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_ft, target), 1):
        x_tr, x_va = train_ft.iloc[tr_idx], train_ft.iloc[va_idx]
        y_tr, y_va = target.iloc[tr_idx], target.iloc[va_idx]

        pre = preprocess.fit(x_tr)
        x_tr_t = pre.transform(x_tr)
        x_va_t = pre.transform(x_va)
        x_te_t = pre.transform(test_ft)

        m = model_factory()
        fit_kwargs = fit_kwargs_factory(x_va_t, y_va) if fit_kwargs_factory else {}
        m.fit(x_tr_t, y_tr, **fit_kwargs)

        va_pred = m.predict_proba(x_va_t)[:, 1]
        fold_aucs.append(roc_auc_score(y_va, va_pred))

        oof[va_idx] += va_pred / N_REPEATS
        pred        += m.predict_proba(x_te_t)[:, 1] / n_total

    if verbose:
        print(f"  폴드 AUC 평균: {np.mean(fold_aucs):.5f} (std {np.std(fold_aucs):.5f})")
        print(f"  OOF AUC      : {roc_auc_score(target, oof):.5f}")

    return oof, pred

In [73]:
lgbm_params_base = dict(
    n_estimators=3000, learning_rate=0.02,
    num_leaves=15, max_depth=3, min_child_samples=15,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.2, reg_lambda=5.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)

lgb_oof_step1, lgb_pred_step1 = run_cv(
    model_factory=lambda: LGBMClassifier(**lgbm_params_base),
    fit_kwargs_factory=lambda xv, yv: dict(
        eval_set=[(xv, yv)], eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
)

submit_step1 = submit_template.copy()
submit_step1["target"] = lgb_pred_step1
submit_step1.to_csv(f"{SUBMIT_PREFIX}_step1.csv", index=False)
print(f"저장됨: {SUBMIT_PREFIX}_step1.csv")

  폴드 AUC 평균: 0.89899 (std 0.01952)
  OOF AUC      : 0.90649
저장됨: submit_머신러닝_이관수_step1.csv


In [74]:
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def objective(trial):
    params = dict(
        n_estimators=3000,
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 8, 64),
        max_depth        = trial.suggest_int("max_depth", 3, 8),
        min_child_samples= trial.suggest_int("min_child_samples", 10, 50),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha",  1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    oof = np.zeros(len(train_ft))
    for tr_idx, va_idx in skf_tune.split(train_ft, target):
        x_tr, x_va = train_ft.iloc[tr_idx], train_ft.iloc[va_idx]
        y_tr, y_va = target.iloc[tr_idx], target.iloc[va_idx]
        pre = preprocess.fit(x_tr)
        x_tr_t, x_va_t = pre.transform(x_tr), pre.transform(x_va)
        m = LGBMClassifier(**params)
        m.fit(x_tr_t, y_tr,
              eval_set=[(x_va_t, y_va)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va_idx] = m.predict_proba(x_va_t)[:, 1]
    return roc_auc_score(target, oof)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print(f"Best 튜닝 OOF AUC: {study.best_value:.5f}")
print("Best params:", study.best_params)

Best 튜닝 OOF AUC: 0.90212
Best params: {'learning_rate': 0.03902889154481096, 'num_leaves': 23, 'max_depth': 5, 'min_child_samples': 32, 'subsample': 0.9593565051094073, 'colsample_bytree': 0.8378291394132933, 'reg_alpha': 3.752213856603954, 'reg_lambda': 9.919823196196006}


In [75]:
lgbm_params_tuned = dict(
    n_estimators=3000, random_state=SEED, n_jobs=-1, verbose=-1,
    **study.best_params
)

print("\n튜닝된 파라미터로 RepeatedKFold 재학습")
lgb_oof_step2, lgb_pred_step2 = run_cv(
    model_factory=lambda: LGBMClassifier(**lgbm_params_tuned),
    fit_kwargs_factory=lambda xv, yv: dict(
        eval_set=[(xv, yv)], eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
)

submit_step2 = submit_template.copy()
submit_step2["target"] = lgb_pred_step2
submit_step2.to_csv(f"{SUBMIT_PREFIX}_step2.csv", index=False)
print(f"저장됨: {SUBMIT_PREFIX}_step2.csv")


튜닝된 파라미터로 RepeatedKFold 재학습
  폴드 AUC 평균: 0.90421 (std 0.01860)
  OOF AUC      : 0.90951
저장됨: submit_머신러닝_이관수_step2.csv


In [76]:

# 1)
lgb_oof = lgb_oof_step2
lgb_pred = lgb_pred_step2
print("\n[LGBM] OOF AUC:", roc_auc_score(target, lgb_oof))

# 2)
print("\n[XGBoost]")
xgb_params = dict(
    n_estimators=3000, learning_rate=0.03, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.2, reg_lambda=2.0,
    random_state=SEED, n_jobs=-1,
    eval_metric="auc", early_stopping_rounds=100, verbosity=0
)
xgb_oof, xgb_pred = run_cv(
    model_factory=lambda: XGBClassifier(**xgb_params),
    fit_kwargs_factory=lambda xv, yv: dict(eval_set=[(xv, yv)], verbose=False)
)

# 3)
print("\n[CatBoost]")
cat_params = dict(
    iterations=3000, learning_rate=0.03, depth=5,
    l2_leaf_reg=3.0, random_seed=SEED,
    eval_metric="AUC", early_stopping_rounds=100, verbose=False
)
cat_oof, cat_pred = run_cv(
    model_factory=lambda: CatBoostClassifier(**cat_params),
    fit_kwargs_factory=lambda xv, yv: dict(eval_set=(xv, yv))
)

# 4) 블렌딩 (단순 평균)
blend_oof  = (lgb_oof  + xgb_oof  + cat_oof ) / 3
blend_pred = (lgb_pred + xgb_pred + cat_pred) / 3

print("\n=== 모델별 OOF AUC ===")
print(f"  LGBM    : {roc_auc_score(target, lgb_oof):.5f}")
print(f"  XGBoost : {roc_auc_score(target, xgb_oof):.5f}")
print(f"  CatBoost: {roc_auc_score(target, cat_oof):.5f}")
print(f"  블렌딩  : {roc_auc_score(target, blend_oof):.5f}")

submit_step3 = submit_template.copy()
submit_step3["target"] = blend_pred
submit_step3.to_csv(f"{SUBMIT_PREFIX}_step3.csv", index=False)
print(f"\n저장됨: {SUBMIT_PREFIX}_step3.csv")


[LGBM] OOF AUC: 0.90951486013986

[XGBoost]
  폴드 AUC 평균: 0.90537 (std 0.01720)
  OOF AUC      : 0.90906

[CatBoost]
  폴드 AUC 평균: 0.90658 (std 0.01523)
  OOF AUC      : 0.90734

=== 모델별 OOF AUC ===
  LGBM    : 0.90951
  XGBoost : 0.90906
  CatBoost: 0.90734
  블렌딩  : 0.91250

저장됨: submit_머신러닝_이관수_step3.csv


In [77]:
print("=" * 60)
print("최종 OOF AUC 비교")
print("=" * 60)

results = {
    f"{SUBMIT_PREFIX}_step1.csv": roc_auc_score(target, lgb_oof_step1),
    f"{SUBMIT_PREFIX}_step2.csv": roc_auc_score(target, lgb_oof_step2),
    f"{SUBMIT_PREFIX}_step3.csv": roc_auc_score(target, blend_oof),
}

best_file = max(results, key=results.get)
for fname, score in results.items():
    flag = "  ← 가장 높음" if fname == best_file else ""
    print(f"  {fname}: {score:.5f}{flag}")

print(f"\n제출 권장 파일: {best_file}")

최종 OOF AUC 비교
  submit_머신러닝_이관수_step1.csv: 0.90649
  submit_머신러닝_이관수_step2.csv: 0.90951
  submit_머신러닝_이관수_step3.csv: 0.91250  ← 가장 높음

제출 권장 파일: submit_머신러닝_이관수_step3.csv


In [78]:
# 위에서 best_file 로 잡힌 파일을 검증
sub = pd.read_csv(best_file)

print(sub.head())
print("shape:", sub.shape)
print("ID 중복 개수:", sub["ID"].duplicated().sum())
print("target 결측 개수:", sub["target"].isna().sum())
print("target 범위:", sub["target"].min(), "~", sub["target"].max())
print("target 평균:", sub["target"].mean())

assert list(sub.columns) == ["ID", "target"]
assert sub["ID"].duplicated().sum() == 0
assert sub["target"].isna().sum() == 0
assert sub["target"].between(0, 1).all()
print("\n이상 없음")

       ID    target
0  test_0  0.010233
1  test_1  0.130856
2  test_2  0.017042
3  test_3  0.101791
4  test_4  0.044158
shape: (896, 2)
ID 중복 개수: 0
target 결측 개수: 0
target 범위: 0.0091336534608118 ~ 0.8828128542644443
target 평균: 0.1237047893793836

이상 없음
